# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described via a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and IDs.
We'll inspect record set `@id`s, field (column) names and their `@id`s for further exploration.

In [ ]:
# List all record sets and their details

record_sets = []
print(f"Available record sets in the dataset:")
for rs in dataset.record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = [field.id for field in rs.fields]
    field_names = [field.name for field in rs.fields]
    print(f"  Fields: {list(zip(field_names, field_ids))}")
    record_sets.append(rs.id)
    print()
if not record_sets:
    print("No record sets found. Please check the schema definition.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Load the first available record set as example (update index if you wish to explore another)
dataframes = {}

if record_sets:
    for rec_id in record_sets:
        print(f"Loading record set with @id: {rec_id}")
        df = pd.DataFrame(list(dataset.records(record_set=rec_id)))
        dataframes[rec_id] = df
        print(f"Loaded {len(df)} records and columns: {df.columns.tolist()}\n")
    # Display head of the first record set
    first_set = record_sets[0]
    print(f"Columns for {first_set}:\n{dataframes[first_set].columns.tolist()}")
    display(dataframes[first_set].head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, and grouping by categorical fields.

**Note:** Update the variable assignments below with specific `@id` values (column names) as identified above.

This example assumes a column such as `cr:abundance_norm_sampleA` (or similar) for numeric analysis and `cr:description` or `cr:sample` for grouping.

In [ ]:
# --- Customize your field `@id`s here based on the previous overview cell! ---

example_record_set_id = record_sets[0] if record_sets else None

# Replace with actual field `@id` or DataFrame column name from previous overview
numeric_field = None
group_field = None
if example_record_set_id:
    columns = dataframes[example_record_set_id].columns.tolist()
    # Try to auto-detect a numeric field
    for col in columns:
        if 'abundance' in col or 'MW' in col or 'count' in col or 'peptide' in col:
            numeric_field = col
            break
    # Try to auto-detect a grouping field
    for col in columns:
        if 'sample' in col or 'description' in col:
            group_field = col
            break
    if not numeric_field:
        # Fallback: use first float column
        for col in columns:
            if pd.api.types.is_numeric_dtype(dataframes[example_record_set_id][col]):
                numeric_field = col
                break
    
    print(f"Using numeric field: {numeric_field}")
    print(f"Using group field: {group_field}\n")
    df = dataframes[example_record_set_id]
    # Drop NA for numeric field
    df_filtered = df.dropna(subset=[numeric_field])
    threshold = df_filtered[numeric_field].mean() if not df_filtered.empty else 0
    filtered_df = df_filtered[df_filtered[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Group by field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (mean of numeric fields):")
        display(grouped_df.head())
else:
    print("No record sets available for EDA.")

## 5. Visualization
Below we plot the distribution of the selected numeric field and mean value per group (if grouping is available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,5))
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        sns.barplot(x=group_means.index.astype(str), y=group_means.values)
        plt.xticks(rotation=45, ha='right')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored the 'Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells' FAIR^2 dataset via its Croissant metadata and record set.

- The dataset contains protein-level measurements and attributes derived from mass spectrometry on mast cell extracellular vesicles.
- We demonstrated how to enumerate record sets and their fields by `@id`, extracted data into pandas DataFrames, filtered and normalized selected numeric fields, performed grouping operations, and visualized the data distributions.
- For a full data catalog, consult the list of fields and inspect the DataFrame columns, using the `@id` references provided by the Croissant schema for reproducibility and downstream ML applications.

Further statistical analysis or machine learning is possible now that the FAIR^2 dataset is loaded and processed in a pandas-friendly format!
